# Portfolio Returns & Realized Vol — Example

Fetch aligned daily returns for a weighted portfolio from yfinance (via the `portfolio_vol` module in this folder), then compute realized portfolio vol over the full fetched sample.

The vol math matches the VBA reference: population covariance (divide by n, `ddof=0`). To change the window, just change `START`/`END` in the config cell.

## 1. One-time setup

Uncomment and run the next cell once to install dependencies into the active Python env. After that you can re-run the notebook without it.

In [1]:
 %pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\kpa32\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 2. Define the portfolio & window

- `WEIGHTS` — dict of ticker → weight. Case-insensitive; must sum to 1.0 (or pass `normalize=True`).
- `START`/`END` — yfinance fetch window. All downstream math runs on the full fetched sample.

In [2]:
import numpy as np
import pandas as pd

from portfolio_vol import Portfolio, fetch_returns

WEIGHTS = {
    "SPY": 0.50,
    "QQQ": 0.30,
    "TLT": 0.20,
}

START = "2024-01-01"
END   = "2025-01-01"

TRADING_DAYS_PER_YEAR = 252  # annualization factor

portfolio = Portfolio.from_weights(WEIGHTS)
portfolio.weights

QQQ    0.3
SPY    0.5
TLT    0.2
dtype: float64

## 3. Fetch daily returns

Log returns, aligned on the intersection of trading days across tickers.

In [4]:
returns = fetch_returns(portfolio, start=START, end=END)
print(f"shape: {returns.shape}   range: {returns.index.min().date()} → {returns.index.max().date()}")
returns.tail()

shape: (251, 3)   range: 2024-01-03 → 2024-12-31


Ticker,QQQ,SPY,TLT
date,,,
2024-12-24,0.013469,0.011054,0.004220
2024-12-26,-0.000680,0.000067,-0.000569
2024-12-27,-0.013382,-0.010582,-0.008232
2024-12-30,-0.013389,-0.011477,0.008005
2024-12-31,-0.008531,-0.003645,-0.005367


## 4. Realized portfolio vol

Matches the Excel/VBA calc over the full sample:
1. Covariance matrix Σ with `ddof=0` (population, divide by n — same as VBA `WorksheetFunction.Covar`).
2. Variance = wᵀ Σ w, vol = √variance, annualized vol = daily_vol × √252.

In [5]:
cov = returns.cov(ddof=0)                # population covariance, matches VBA Covar
w   = portfolio.weights.loc[cov.columns]  # reindex so rows/cols/weights align

# Step 1: Σw  (the "intermediate result" column in the Excel sheet)
sigma_w = cov @ w

# Step 2: wᵀ Σ w  (scalar variance)
daily_var = float(w @ sigma_w)

# Step 3: sqrt
daily_vol = np.sqrt(daily_var)
ann_vol   = daily_vol * np.sqrt(TRADING_DAYS_PER_YEAR)

print(f"sample: {len(returns)} days  ({returns.index.min().date()} → {returns.index.max().date()})")
print(f"daily portfolio vol:      {daily_vol:.4%}")
print(f"annualized portfolio vol: {ann_vol:.2%}")

lookback window:    63 days  (2024-10-02 → 2024-12-31)
daily portfolio vol:     0.7302%
annualized portfolio vol: 11.59%


In [6]:
# Sanity check: same number via the portfolio return series.
# Uses population stdev (ddof=0) to stay consistent with the matrix path.
port_ret = returns @ w
daily_vol_check = port_ret.std(ddof=0)
print(f"via port_ret.std(ddof=0): {daily_vol_check:.6%}")
print(f"via wᵀ Σ w:               {daily_vol:.6%}")
assert np.isclose(daily_vol, daily_vol_check), "vol paths disagree"

via port_ret.std(ddof=0): 0.730194%
via wᵀ Σ w:               0.730194%


## 5. Component vol & correlation matrix

Same full sample. Weighted-average component vol is always ≥ portfolio vol — their ratio is a crude diversification measure.

In [10]:
# Per-name daily vol (sqrt of the diagonal of the cov matrix)
component_daily_vol = pd.Series(np.sqrt(np.diag(cov)), index=cov.columns, name="daily_vol")
component_ann_vol   = component_daily_vol * np.sqrt(TRADING_DAYS_PER_YEAR)

# Weighted-average component vol (ignores correlation)
wavg_daily_vol = float((w * component_daily_vol).sum())
wavg_ann_vol   = wavg_daily_vol * np.sqrt(TRADING_DAYS_PER_YEAR)



print(f"weighted-avg component vol — daily: {wavg_daily_vol:.4%}   annualized: {wavg_ann_vol:.2%}")
print(f"portfolio vol              — daily: {daily_vol:.4%}   annualized: {ann_vol:.2%}")
print(f"diversification ratio (wavg / port): {wavg_daily_vol / daily_vol:.3f}")

#compute avg correlation
avg_corr = (ann_vol**2)/wavg_ann_vol**2
print(f"average correlation: {avg_corr:.3f}")

pd.DataFrame({"weight": w, "daily_vol": component_daily_vol, "ann_vol": component_ann_vol})

weighted-avg component vol — daily: 0.8972%   annualized: 14.24%
portfolio vol              — daily: 0.7302%   annualized: 11.59%
diversification ratio (wavg / port): 1.229
average correlation: 0.662


,weight,daily_vol,ann_vol
Ticker,,,
QQQ,0.3,0.010733,0.170377
SPY,0.5,0.007859,0.124752
TLT,0.2,0.009115,0.144693


In [8]:
# Correlation matrix (invariant to ddof, so this matches either convention)
returns.corr()

Ticker,QQQ,SPY,TLT
Ticker,,,
QQQ,1.000000,0.934422,0.011326
SPY,0.934422,1.000000,0.028722
TLT,0.011326,0.028722,1.000000


## 6. Error handling — what to expect

- `PortfolioError` — bad spec (weights don't sum to 1, duplicate ticker, non-finite weight, empty dict).
- `FetchError` — yfinance issue (bad ticker, no overlap across names, all retries failed, too few observations after alignment).

Try swapping in a bogus ticker below to see the error path.

In [9]:
from portfolio_vol import FetchError, PortfolioError

try:
    bad = Portfolio.from_weights({"SPY": 0.5, "NOTATICKER": 0.5})
    fetch_returns(bad, start=START, end=END)
except (FetchError, PortfolioError) as e:
    print(f"{type(e).__name__}: {e}")

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: NOTATICKER"}}}
$NOTATICKER: possibly delisted; no timezone found

1 Failed download:
['NOTATICKER']: possibly delisted; no timezone found


FetchError: yfinance returned no data for: ['NOTATICKER']. Check the ticker symbols and date range.
